# D17 - Loan Disbursement Trend Analysis
ABES Engineering College | MCA 2nd Year | Data Analyst

Synthetic-only exploratory analysis. One row is a transaction event; a successful disbursement is an explicit disbursement event with successful status.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import plotly.express as px

ROOT = Path.cwd()
if not (ROOT / 'loan_app').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from loan_app.data import generate_dataset, validate_and_normalize
from loan_app.analytics import calculate_kpis, monthly_summary, category_summary


## Load and profile the synthetic source

In [ ]:
sample = ROOT / 'data' / 'loan_transactions_sample.csv'
raw = pd.read_csv(sample) if sample.exists() else generate_dataset(rows=12000, seed=42)
result = validate_and_normalize(raw)
df, quality = result.data, result.quality
print(quality)
df.head()

## Data quality and descriptive statistics
Generated rows are fictional and not calibrated to any lender. Validation removes duplicate IDs and invalid data; inspect the quality counts before interpreting aggregates.

In [ ]:
df.dtypes


In [ ]:
df[['amount']].describe(percentiles=[.25,.5,.75,.95]).T

## KPI definitions
Successful disbursement means `event_type == 'disbursement'` and `status == 'success'`. Transaction success rate uses all event types and is different from disbursement-attempt success rate.

In [ ]:
kpis = calculate_kpis(df)
pd.Series(kpis)

## Monthly and category trends

In [ ]:
monthly = monthly_summary(df)
fig = px.line(monthly, x='month', y='disbursed_amount', markers=True, title='Monthly successful synthetic disbursed amount')
fig.update_yaxes(title='INR')
fig.show()

In [ ]:
by_type = category_summary(df, 'loan_type')
by_type

In [ ]:
px.bar(by_type, x='loan_type', y='disbursed_amount', title='Successful synthetic disbursed amount by loan type').show()

## Interpretation and limitations
Discuss descriptive shape only. Generated patterns are generator behavior, not actual market evidence. A reversal is not netted from gross disbursement totals in this MVP. The sample delinquency flag has no aging definition and is not a portfolio delinquency rate. Confirm event mapping and currency with a real business owner before any future use with observed data.